# Stage 3: Implementing an RAG System for Question Answering

## Libraries

In [2]:
!pip install pandas numpy seaborn matplotlib

In [3]:
import os
import re
import html
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from collections import Counter
from openai import OpenAI
import json
import time
from google.colab import userdata
from bs4 import BeautifulSoup



## Environment setup

In [4]:
ENV = "colab"

In [5]:
if ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


You also need your own `OPENAI_API_KEY`. Add it to Colab Secrets.

##

## Data preparation
In our previous stages we heavily normalized and cleaned our content columns. For RAG we need to preserve punctuation, but we still need to remove duplicates, strip html or urls.  So we load our initial dataset `CLT/ai_media_dataset_20250911.csv` (should be already in your folder from previous stages) and do a specific cleaning for later RAG implementation.

To keep your files organazied in your Google Drive create another folder `"stage_3"` inside `CLT`.

In [ ]:
# -----------------------------
# 1) Load raw dataset
# -----------------------------
if ENV == "colab":
    raw_path = "/content/drive/My Drive/CLT/ai_media_dataset_20250911.csv"
else:
    raw_path = "data/ai_media_dataset_20250911.csv"

df = pd.read_csv(raw_path, encoding="utf-8", engine="python")

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())


# -----------------------------
# 3) Restore tags as list (IMPORTANT)
# -----------------------------
def restore_tags(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except:
        return []

df['tags'] = df['tags'].apply(restore_tags)


# -----------------------------
# 4) Cleaning functions (RAG-safe)
# -----------------------------
def clean_text(text):
    text = str(text) if pd.notna(text) else ""

    # decode HTML entities
    text = html.unescape(text)

    # remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text(" ")

    # remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # normalize unicode quotes/dashes
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "-")

    # keep punctuation, remove weird junk
    text = re.sub(r"[^\w\s\.,;:!?\-\'\"()/]", " ", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_title(text):
    text = str(text) if pd.notna(text) else ""
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


# -----------------------------
# 5) Apply cleaning
# -----------------------------
df['title'] = df['title'].apply(clean_title)
df['content'] = df['content'].apply(clean_text)


# -----------------------------
# 6) Remove bad rows
# -----------------------------
df = df.dropna(subset=['title', 'content'])

df = df[
    (df['title'].str.strip() != "") &
    (df['content'].str.strip() != "")
].copy()

# remove duplicates
df = df.drop_duplicates(subset=['title', 'content']).reset_index(drop=True)

print("Cleaned shape:", df.shape)


# -----------------------------
# 7) Optional: length filter (recommended)
# -----------------------------
df['word_count'] = df['content'].str.split().str.len()

# remove extremely short articles
df = df[df['word_count'] > 50].copy()

print("After length filter:", df.shape)


# -----------------------------
# 8) Drop helper column
# -----------------------------
df = df.drop(columns=['word_count'])


# -----------------------------
# 9) Save cleaned file
# -----------------------------
if ENV == "colab":
    save_path = "/content/drive/My Drive/CLT/stage_3/clean_title_content.csv"
else:
    save_path = "data/stage_3/clean_title_content.csv"

df.to_csv(save_path, index=False, encoding="utf-8")

print("Saved cleaned dataset to:", save_path)

# 1. Q&A Dataset Construction
##1.1 Data Loading


We load `clean_title_content.csv`  with columns `['date', 'domain', 'url', 'tags', 'title', 'content']`. Its clean, deduplicated and we removed special characters.

In [ ]:
if ENV == "colab":
    norm_path = '/content/drive/My Drive/CLT/stage_3/clean_title_content.csv'
else:
    norm_path = 'data/stage_3/clean_title_content.csv'

df = pd.read_csv(norm_path, encoding='utf-8')

# Restore list columns serialized as strings by CSV
list_cols = ['tags']   # add more columns here if needed, e.g. ['tags', 'entities']

for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: ast.literal_eval(x) if pd.notna(x) else []
        )


print("Loaded shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 3 rows:")
print(df.head(3))

## 1.2 Searching for most frequent tags
In the following code block we are searching for mostfrequent tags

In [ ]:
# Flatten list of tags
all_tags = [tag.lower() for tags in df['tags'] for tag in tags]

# Count frequencies
tag_counter = Counter(all_tags)

# Convert to DataFrame
tag_df = (
    pd.DataFrame(tag_counter.items(), columns=['tag', 'count'])
    .sort_values(by='count', ascending=False)
    .reset_index(drop=True)
)

print("Number of unique tags:", len(tag_df))
print(tag_df.head(20))

Now we would search for text passages in content column which include this topics.

##1.3 Assign topic/s to each document

In [ ]:
# =========================
# 2) Topic groups from your tags
# =========================
topic_groups = {
    "generative_ai": [
        "generativeai", "largelanguagemodel", "chatgpt", "openai", "gemini"
    ],
    "nlp": [
        "naturallanguageprocessing", "conversationalai"
    ],
    "robotics_autonomy": [
        "robotics", "autonomousdriving"
    ],
    "ethics_governance": [
        "accountability", "militaryanddefense"
    ],
    "technical_ai": [
        "deeplearning", "finetuning", "gpu"
    ],
    "reasoning_planning": [
        "reasoning", "planning"
    ],
    "applications": [
        "personalisation", "video", "audio"
    ],
    "emerging_tech": [
        "quantumcomputing"
    ]
}

# =========================
# 3) Assign each document to one or more topic groups
# =========================
def assign_topic_groups(tags):
    tags_lower = {str(t).strip().lower() for t in tags}
    matched = []
    for group, group_tags in topic_groups.items():
        if any(tag in tags_lower for tag in group_tags):
            matched.append(group)
    return matched

df['topic_groups'] = df['tags'].apply(assign_topic_groups)

# Keep only rows that match at least one topic group
df_topics = df[df['topic_groups'].apply(len) > 0].copy()

print("Rows with matched topic groups:", df_topics.shape[0])


##1.4 Split content into sentences and parapgraphs, create scores for passages including the topic

In [ ]:
# =========================
# 4) Passage extraction from content
#    - prefer paragraphs
#    - fallback to sentence windows
# =========================
def normalize_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def split_paragraphs(text):
    # split on blank lines or paragraph-like breaks
    raw_parts = re.split(r'\n\s*\n|\r\n\s*\r\n', str(text))
    parts = [re.sub(r'\s+', ' ', p).strip() for p in raw_parts]
    return [p for p in parts if p]

def split_sentences(text):
    # simple sentence splitter
    text = re.sub(r'\s+', ' ', str(text)).strip()
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

def score_passage(passage):
    """
    Higher score = better candidate for QA generation.
    Rewards informative, self-contained passages.
    """
    p = str(passage)
    word_count = len(p.split())

    # base length preference: target around 80-220 words
    if word_count < 40:
        length_score = -100
    elif 80 <= word_count <= 220:
        length_score = 30
    elif 40 <= word_count < 80 or 220 < word_count <= 300:
        length_score = 15
    else:
        length_score = 0

    # reward informative features
    feature_score = 0
    if re.search(r'\b(ai|artificial intelligence|model|models|robot|policy|regulation|ethics|openai|google|microsoft|meta|nvidia|chatgpt|llm)\b', p, re.I):
        feature_score += 10
    if re.search(r'\b\d{4}\b', p):  # year mention
        feature_score += 3
    if re.search(r'\b(because|however|while|although|but|therefore|raises|improves|reduces|enables|compared to)\b', p, re.I):
        feature_score += 6

    # penalize noisy passages
    noise_score = 0
    if p.count('|') > 2:
        noise_score -= 10
    if len(re.findall(r'[^\w\s.,;:!?()\'"-]', p)) > 20:
        noise_score -= 10

    return length_score + feature_score + noise_score

def extract_best_passage(text, min_words=60, max_words=220):
    """
    Try paragraph-based extraction first.
    If no good paragraph, fallback to sentence-window extraction.
    """
    text = str(text).strip()
    if not text:
        return None

    # paragraph-first strategy
    paragraphs = split_paragraphs(text)
    candidate_paragraphs = []

    for p in paragraphs:
        wc = len(p.split())
        if min_words <= wc <= max_words:
            candidate_paragraphs.append(p)

    if candidate_paragraphs:
        best = max(candidate_paragraphs, key=score_passage)
        return normalize_text(best)

    # if paragraphs too long/short, fallback to sentence windows
    sentences = split_sentences(text)
    if not sentences:
        return None

    windows = []
    n = len(sentences)

    # try windows of 3-6 sentences
    for window_size in range(3, 7):
        for i in range(n - window_size + 1):
            chunk = " ".join(sentences[i:i + window_size]).strip()
            wc = len(chunk.split())
            if min_words <= wc <= max_words:
                windows.append(chunk)

    if windows:
        best = max(windows, key=score_passage)
        return normalize_text(best)

    # last fallback: first max_words words if long enough
    words = text.split()
    if len(words) >= min_words:
        return " ".join(words[:max_words])

    return None

## 1.5 Create a balanced sample of passages by topic



In [ ]:
# =========================
# 5) Balanced document sampling by topic group
# =========================
TARGET_TOTAL = 80
TOPIC_NAMES = list(topic_groups.keys())
PER_TOPIC = max(6, TARGET_TOTAL // len(TOPIC_NAMES))

samples = []

for topic in TOPIC_NAMES:
    subset = df_topics[df_topics['topic_groups'].apply(lambda groups: topic in groups)].copy()
    if subset.empty:
        continue

    # shuffle for reproducibility
    subset = subset.sample(frac=1, random_state=42)

    # oversample candidates because some passages may fail extraction
    subset = subset.head(PER_TOPIC * 3).copy()
    subset['sampled_topic'] = topic
    samples.append(subset)

candidate_docs = pd.concat(samples).drop_duplicates(subset=['title', 'content']).reset_index(drop=True)

print("Candidate docs:", candidate_docs.shape)

# =========================
# 6) Extract one representative passage per sampled document
# =========================
candidate_docs['passage'] = candidate_docs['content'].apply(extract_best_passage)
candidate_docs['passage_word_count'] = candidate_docs['passage'].apply(lambda x: len(str(x).split()) if pd.notna(x) and x else 0)

selected = candidate_docs.dropna(subset=['passage']).copy()
selected = selected[selected['passage_word_count'] >= 60].copy()

print("Docs with extracted passages:", selected.shape)
# =========================
# 7) Re-balance final selection by topic
# =========================
final_parts = []

for topic in TOPIC_NAMES:
    subset = selected[selected['sampled_topic'] == topic].copy()
    if subset.empty:
        continue

    n = min(PER_TOPIC, len(subset))
    final_parts.append(subset.head(n))

selected_passages = pd.concat(final_parts).drop_duplicates(subset=['title', 'passage']).reset_index(drop=True)

# Trim to target total if needed
if len(selected_passages) > TARGET_TOTAL:
    selected_passages = selected_passages.sample(n=TARGET_TOTAL, random_state=42).reset_index(drop=True)

# Add ids
selected_passages.insert(0, 'passage_id', [f"p_{i:03d}" for i in range(1, len(selected_passages) + 1)])

# Keep useful columns
keep_cols = [
    'passage_id',
    'title',
    'sampled_topic',
    'tags',
    'passage',
    'passage_word_count'
]

# include optional metadata if present
for extra_col in ['date', 'source', 'url']:
    if extra_col in selected_passages.columns:
        keep_cols.insert(keep_cols.index('tags'), extra_col)

selected_passages = selected_passages[keep_cols].copy()

print("Final selected passages:", selected_passages.shape)
print(selected_passages['sampled_topic'].value_counts())
print(selected_passages[['passage_id', 'sampled_topic', 'title', 'passage_word_count']].head())



## 1.6 Save selected passages

In [ ]:
# =========================
# 8) Save result
# =========================
if ENV == "colab":
    save_path = '/content/drive/My Drive/CLT/stage_3/selected_passages.csv'
else:
    save_path = 'data/stage_3/selected_passages.csv'

selected_passages.to_csv(save_path, index=False, encoding='utf-8')
print("Saved to:", save_path)


# =========================
# 9) Optional: inspect a few passages
# =========================
for _, row in selected_passages.head(3).iterrows():
    print("\n" + "="*80)
    print("PASSAGE ID:", row['passage_id'])
    print("TOPIC:", row['sampled_topic'])
    print("TITLE:", row['title'])
    print("TAGS:", row['tags'])
    print("PASSAGE:", row['passage'][:800], "...")

## 1.7. Generate 200 questions with LLM
🟡🟡🟡🟡 Alla note: To not spend too much money for OPEN_API_KEY for every time generating questions, I am generating atm only for 5 passages. I will adjust that towards the end of the project


In [ ]:
# ============================================================
# GENERATE TOPIC-ALIGNED QUESTIONS FROM SELECTED PASSAGES
# - grounded in the "passage" column
# - guided by topic/title/tags
# - supports small test runs before full generation
# - robust JSON parsing + retries + intermediate saving
# ============================================================


# -----------------------------
# 1) OpenAI client (Colab)
# -----------------------------
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Add it in Colab Secrets.")

client = OpenAI(api_key=OPENAI_API_KEY)

# -----------------------------
# 2) Load selected passages
# -----------------------------
if ENV == "colab":
    passages_path = "/content/drive/My Drive/CLT/stage_3/selected_passages.csv"
else:
    passages_path = "data/stage_3/selected_passages.csv"

df = pd.read_csv(passages_path, encoding="utf-8", engine="python")

print("Loaded passages shape:", df.shape)
print("Columns:", df.columns.tolist())

# -----------------------------
# 3) Restore list columns if CSV serialized them as strings
# -----------------------------
def restore_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

for col in ["tags"]:
    if col in df.columns:
        df[col] = df[col].apply(restore_list)

# Add passage_id if missing
if "passage_id" not in df.columns:
    df = df.reset_index(drop=True)
    df.insert(0, "passage_id", [f"p_{i:03d}" for i in range(1, len(df) + 1)])

# Basic checks
required_cols = ["passage"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Optional metadata columns
if "title" not in df.columns:
    df["title"] = None
if "sampled_topic" not in df.columns:
    df["sampled_topic"] = None
if "tags" not in df.columns:
    df["tags"] = [[] for _ in range(len(df))]

# Drop empty passages
df["passage"] = df["passage"].astype(str).str.strip()
df = df[df["passage"] != ""].copy().reset_index(drop=True)

print("Usable passages:", df.shape)

# -----------------------------
# 4) SETTINGS
# -----------------------------
TEST_MODE = True           # True = small test, False = full generation
TEST_N_PASSAGES = 5        # how many passages to use in test mode
QUESTIONS_PER_PASSAGE = 4  # 50 passages x 4 = 200 questions
RANDOM_SEED = 42
MODEL_NAME = "gpt-4o-mini"
MAX_RETRIES = 3
SLEEP_SECONDS = 1.0
SAVE_EVERY = 5             # save intermediate files every N passages
VERBOSE = True             # print raw model output in test mode

# Select subset
if TEST_MODE:
    run_df = df.sample(n=min(TEST_N_PASSAGES, len(df)), random_state=RANDOM_SEED).copy()
    print(f"TEST MODE ON -> {len(run_df)} passages")
else:
    run_df = df.copy()
    print(f"FULL MODE ON -> {len(run_df)} passages")

# -----------------------------
# 5) Prompt builder
# -----------------------------
def build_prompt(passage, title=None, topic=None, tags=None, questions_per_passage=4):
    tags_text = ", ".join(tags) if isinstance(tags, list) else str(tags) if tags is not None else "N/A"
    title_text = title if title else "N/A"
    topic_text = topic if topic else "N/A"

    return f"""
You are creating a high-quality question-answer dataset for evaluating a retrieval-augmented generation (RAG) system about AI trends.

Context metadata:
- Title: {title_text}
- Topic group: {topic_text}
- Tags: {tags_text}

Task:
Generate EXACTLY {questions_per_passage} question-answer pairs based ONLY on the passage below.

Requirements:
- Return ONLY valid JSON
- Do not use markdown
- Do not add explanations, comments, or notes
- Output must be a JSON array with exactly {questions_per_passage} items
- Each item must contain exactly these keys:
  - "question"
  - "answer"
  - "type"

Question design rules:
- Every question must be answerable ONLY from the passage
- Questions must be clearly related to the AI topic, tags
- Prefer questions about AI systems, models, methods, risks, governance, regulation, applications, companies, trends, or comparisons mentioned in the text
- Avoid generic media questions unrelated to the AI content
- Avoid trivial wording unless the fact is important
- Keep answers concise but complete
- Use realistic phrasing suitable for evaluating a question-answering system

Question types:
- "factual": asks for a directly stated fact
- "analytical": asks for a reason, implication, explanation, or interpretation supported by the passage
- "comparative": compares two items, approaches, situations, or time points explicitly mentioned in the passage

Type distribution:
- Include at least 1 factual question
- Include at least 1 analytical question
- Include 1 comparative question IF the passage clearly supports it
- If comparative is not supported, replace it with another factual or analytical question

Output format:
[
  {{
    "question": "What is ...?",
    "answer": "...",
    "type": "factual"
  }},
  {{
    "question": "Why does ...?",
    "answer": "...",
    "type": "analytical"
  }}
]

Passage:
\"\"\"{passage}\"\"\"
"""

# -----------------------------
# 6) JSON parsing helpers
# -----------------------------
def extract_json_array(text):
    """
    Extract the first valid JSON array from model output.
    Handles:
    - plain JSON
    - fenced JSON
    - extra text before/after JSON
    """
    if text is None:
        raise ValueError("Model returned None")

    text = text.strip()
    if not text:
        raise ValueError("Model returned empty text")

    # Remove code fences if present
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Try direct parse
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass

    # Fallback: extract first JSON array substring
    match = re.search(r"\[\s*\{.*\}\s*\]", text, flags=re.DOTALL)
    if match:
        parsed = json.loads(match.group(0))
        if isinstance(parsed, list):
            return parsed

    raise ValueError("Could not parse a valid JSON array from model output.")

def validate_qa_item(item):
    if not isinstance(item, dict):
        return False

    needed = {"question", "answer", "type"}
    if not needed.issubset(item.keys()):
        return False

    if not isinstance(item["question"], str) or not item["question"].strip():
        return False
    if not isinstance(item["answer"], str) or not item["answer"].strip():
        return False
    if item["type"] not in {"factual", "analytical", "comparative"}:
        return False

    return True

def clean_qa_list(qa_list, expected_n):
    cleaned = []
    for item in qa_list:
        if validate_qa_item(item):
            cleaned.append({
                "question": item["question"].strip(),
                "answer": item["answer"].strip(),
                "type": item["type"].strip().lower()
            })
    return cleaned[:expected_n]

# -----------------------------
# 7) Single-passage generation with retries
# -----------------------------
def generate_qa_for_passage(
    passage,
    title=None,
    topic=None,
    tags=None,
    questions_per_passage=4,
    model="gpt-4o-mini",
    max_retries=3,
    sleep_seconds=1.0,
    verbose=False
):
    prompt = build_prompt(
        passage=passage,
        title=title,
        topic=topic,
        tags=tags,
        questions_per_passage=questions_per_passage
    )

    last_error = None
    last_output = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "You output only valid JSON arrays. No markdown. No explanations."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.3
            )

            output_text = response.choices[0].message.content
            last_output = output_text

            if verbose:
                print(f"\n--- RAW OUTPUT attempt {attempt} ---")
                print(repr(output_text[:2000]))
                print("------------------------------------")

            qa_list = extract_json_array(output_text)
            qa_list = clean_qa_list(qa_list, questions_per_passage)

            if len(qa_list) == questions_per_passage:
                return qa_list, output_text

            last_error = ValueError(
                f"Model returned {len(qa_list)} valid items, expected {questions_per_passage}."
            )

        except Exception as e:
            last_error = e

        time.sleep(sleep_seconds)

    raise RuntimeError(
        f"Failed after {max_retries} retries. Last error: {last_error}. Last output: {repr(last_output)}"
    )

# -----------------------------
# 8) Full dataset generation
# -----------------------------
def generate_qa_dataset(
    passages_df,
    questions_per_passage=4,
    model="gpt-4o-mini",
    max_retries=3,
    sleep_seconds=1.0,
    save_every=10,
    verbose=False,
    output_csv_path=None,
    error_csv_path=None
):
    qa_rows = []
    error_rows = []

    total = len(passages_df)

    for idx, row in passages_df.reset_index(drop=True).iterrows():
        passage_id = row["passage_id"]
        passage = row["passage"]
        title = row["title"]
        topic = row["sampled_topic"]
        tags = row["tags"]

        try:
            qa_list, raw_output = generate_qa_for_passage(
                passage=passage,
                title=title,
                topic=topic,
                tags=tags,
                questions_per_passage=questions_per_passage,
                model=model,
                max_retries=max_retries,
                sleep_seconds=sleep_seconds,
                verbose=verbose
            )

            for qa in qa_list:
                qa_rows.append({
                    "passage_id": passage_id,
                    "title": title,
                    "sampled_topic": topic,
                    "tags": tags,
                    "passage": passage,
                    "question": qa["question"],
                    "answer_gold": qa["answer"],
                    "type": qa["type"]
                })

            print(f"[{idx+1}/{total}] OK - {passage_id} -> {len(qa_list)} questions")

        except Exception as e:
            error_rows.append({
                "passage_id": passage_id,
                "title": title,
                "sampled_topic": topic,
                "error": str(e),
                "passage_preview": passage[:300]
            })
            print(f"[{idx+1}/{total}] ERROR - {passage_id}: {e}")

        # Intermediate save
        if save_every and (idx + 1) % save_every == 0:
            if output_csv_path:
                pd.DataFrame(qa_rows).to_csv(output_csv_path, index=False, encoding="utf-8")
            if error_csv_path:
                pd.DataFrame(error_rows).to_csv(error_csv_path, index=False, encoding="utf-8")
            print(f"Intermediate save after {idx+1} passages.")

    qa_df = pd.DataFrame(qa_rows)
    error_df = pd.DataFrame(error_rows)

    if output_csv_path:
        qa_df.to_csv(output_csv_path, index=False, encoding="utf-8")
    if error_csv_path:
        error_df.to_csv(error_csv_path, index=False, encoding="utf-8")

    return qa_df, error_df

# -----------------------------
# 9) Output paths
# -----------------------------
suffix = "test" if TEST_MODE else "full"

if ENV == "colab":
    qa_output_path = f"/content/drive/My Drive/CLT/stage_3/qa_dataset_{suffix}.csv"
    qa_error_path = f"/content/drive/My Drive/CLT/stage_3/qa_errors_{suffix}.csv"
    qa_final_path = f"/content/drive/My Drive/CLT/stage_3/qa_dataset_{suffix}_final.csv"
else:
    qa_output_path = f"data/stage_3/qa_dataset_{suffix}.csv"
    qa_error_path = f"data/stage_3/qa_errors_{suffix}.csv"
    qa_final_path = f"data/stage_3/qa_dataset_{suffix}_final.csv"

# -----------------------------
# 10) Run generation
# -----------------------------
qa_df, error_df = generate_qa_dataset(
    passages_df=run_df,
    questions_per_passage=QUESTIONS_PER_PASSAGE,
    model=MODEL_NAME,
    max_retries=MAX_RETRIES,
    sleep_seconds=SLEEP_SECONDS,
    save_every=SAVE_EVERY,
    verbose=VERBOSE if TEST_MODE else False,
    output_csv_path=qa_output_path,
    error_csv_path=qa_error_path
)

print("\nGeneration finished.")
print("QA shape:", qa_df.shape)
print("Error shape:", error_df.shape)

# -----------------------------
# 11) Deduplicate and optionally cap final size
# -----------------------------
if not qa_df.empty:
    qa_df = qa_df.drop_duplicates(subset=["question"]).reset_index(drop=True)

    # For full mode, optionally cap to exactly 200
    TARGET_QA_TOTAL = 200
    if not TEST_MODE and len(qa_df) > TARGET_QA_TOTAL:
        qa_df = qa_df.head(TARGET_QA_TOTAL).copy()

    qa_df.to_csv(qa_final_path, index=False, encoding="utf-8")

    print("\nFinal deduplicated QA shape:", qa_df.shape)
    print("Saved final QA file:", qa_final_path)

    print("\nQuestion type distribution:")
    print(qa_df["type"].value_counts(dropna=False))

    print("\nSample questions:")
    print(qa_df[["question", "answer_gold", "type"]].head(10))

if not error_df.empty:
    print("\nErrors preview:")
    print(error_df.head())

# -----------------------------
# 12) Useful presets
# -----------------------------
# Small cheap test:
# TEST_MODE = True
# TEST_N_PASSAGES = 5
# QUESTIONS_PER_PASSAGE = 4

# Full run:
# TEST_MODE = False
# QUESTIONS_PER_PASSAGE = 4
# -> with 50 passages, this gives about 200 questions before deduplication

# -----------------------------
# 13) Optional helper: preview passages before generation
# -----------------------------
def preview_passages(passages_df, n=3):
    cols = [c for c in ["passage_id", "title", "sampled_topic", "tags", "passage"] if c in passages_df.columns]
    sample = passages_df[cols].head(n)

    for _, row in sample.iterrows():
        print("\n" + "=" * 80)
        print("PASSAGE ID:", row.get("passage_id"))
        print("TITLE:", row.get("title"))
        print("TOPIC:", row.get("sampled_topic"))
        print("TAGS:", row.get("tags"))
        print("PASSAGE:", str(row.get("passage"))[:1000], "...")

# Example:
# preview_passages(run_df, n=2)